# SIPREBAR — Pipeline Prediksi EBT Sulawesi Selatan
### Dari Data Mentah hingga Evaluasi Model

**Penelitian Skripsi** — UIN Alauddin Makassar, Teknik Informatika, 2026
*Prediksi Produksi Energi Baru Terbarukan Sektor Kelistrikan Menggunakan LSTM*

---

Notebook ini memuat **seluruh alur penelitian** sebelum masuk ke sistem web (frontend & backend):

| Tahap | Isi |
|---|---|
| 1 | Data tahunan Dinas ESDM Sulsel (angka terkunci) |
| 2 | Penentuan titik koordinat cuaca per kategori pembangkit |
| 3 | Penarikan data cuaca riil dari **NASA POWER** |
| 4 | **Disagregasi** tahunan → bulanan (dataset utuh terbentuk) |
| 5 | Rekayasa fitur & framing target (Capacity Factor) |
| 6 | **Walk-forward validation** untuk memilih konfigurasi |
| 7 | Pelatihan model produksi (pooled + embedding + ensembling) |
| 8 | Benchmark metode tradisional (ARIMA, naive, seasonal naive) |
| 9 | **Evaluasi akhir** & tabel perbandingan |

> **Cara pakai di Google Colab:** `File → Upload notebook`, lalu jalankan sel berurutan dari atas ke bawah (`Runtime → Run all`). Notebook ini **mandiri** — tidak butuh file dari repositori, kecuali bila Anda memilih memakai dataset yang sudah jadi di Tahap 4.

---
## Tahap 0 — Persiapan Lingkungan

Colab sudah menyediakan `pandas`, `numpy`, `tensorflow`, dan `scikit-learn`. Yang perlu dipastikan hanya `requests` (untuk NASA POWER) dan `statsmodels` (untuk ARIMA).

In [ ]:
!pip install -q requests statsmodels

import os, json, warnings
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import tensorflow as tf

# Seed global. Semua angka di notebook ini direproduksi dari seed yang sama.
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow :", tf.__version__)
print("Pandas     :", pd.__version__)
print("GPU        :", "tersedia" if tf.config.list_physical_devices('GPU') else "tidak ada (CPU saja, cukup untuk dataset ini)")

---
## Tahap 1 — Data Tahunan Dinas ESDM Sulawesi Selatan

Sumber primer penelitian: sheet *Worksheet* pada berkas
`PERHITUNGAN_BAURAN_ENERGI_ESDM` tahun 2023, 2024, dan 2025.

Cakupan **5 jenis PLT**: PLTA, PLTM, PLTS, PLTS Atap, PLTB.
PLTMH dan PLT Hybrid dikeluarkan dari ruang lingkup (dokumentasi historisnya belum konsisten).

> ### ⚠️ Keterbatasan yang WAJIB diungkap di laporan
> Kolom **Produksi** dari sumber ESDM **bukan hasil pengukuran meter**, melainkan hasil kalkulasi:
>
> $$\text{Produksi} = \text{Kapasitas} \times \text{CF asumsi} \times 8760\ \text{jam}$$
>
> Konsekuensinya, kolom Produksi mewarisi pola dari Kapasitas dan asumsi *Capacity Factor*. Ini keterbatasan yang diwariskan dari sumber data primer — harus dinyatakan terbuka di BAB IV/V, bukan disembunyikan.
>
> Data 2025 yang dipakai adalah **versi yang sudah diperbaiki** Dinas ESDM (label tahun benar, angka genuinely berbeda dari 2024, periode Januari–Desember penuh).

In [ ]:
# Angka tahunan TERKUNCI per jenis PLT (Produksi dalam GWh, Kapasitas dalam MW).
ANNUAL = {
    2023: {
        "PLTA":      {"Produksi": 3434.270400, "Kapasitas": 653.400000},
        "PLTM":      {"Produksi":  337.592880, "Kapasitas":  64.230000},
        "PLTS":      {"Produksi":    9.005280, "Kapasitas":   5.140000},
        "PLTS Atap": {"Produksi":    0.333756, "Kapasitas":   0.190500},
        "PLTB":      {"Produksi":  569.400000, "Kapasitas": 130.000000},
    },
    2024: {
        "PLTA":      {"Produksi": 3729.657600, "Kapasitas": 709.600000},
        "PLTM":      {"Produksi":  350.207280, "Kapasitas":  66.630000},
        "PLTS":      {"Produksi":    9.005280, "Kapasitas":   5.140000},
        "PLTS Atap": {"Produksi":   12.186036, "Kapasitas":   6.955500},
        "PLTB":      {"Produksi":  626.340000, "Kapasitas": 143.000000},
    },
    2025: {
        "PLTA":      {"Produksi": 4155.575468, "Kapasitas": 790.634602},
        "PLTM":      {"Produksi":  363.293017, "Kapasitas":  69.119676},
        "PLTS":      {"Produksi":   10.975930, "Kapasitas":   6.264800},
        "PLTS Atap": {"Produksi":   13.160919, "Kapasitas":   7.511940},
        "PLTB":      {"Produksi":  688.974000, "Kapasitas": 157.300000},
    },
}

JENIS_PLT = ["PLTA", "PLTB", "PLTM", "PLTS", "PLTS Atap"]

ringkas = pd.DataFrame([
    {"Tahun": th, "Jenis": j, "Produksi (GWh)": v["Produksi"], "Kapasitas (MW)": v["Kapasitas"]}
    for th, d in ANNUAL.items() for j, v in d.items()
])
print("Total produksi EBT per tahun (GWh):")
print(ringkas.groupby("Tahun")["Produksi (GWh)"].sum().round(2).to_string())
ringkas.pivot(index="Jenis", columns="Tahun", values="Produksi (GWh)").round(2)

---
## Tahap 2 — Titik Koordinat & Parameter Cuaca

Tiap kategori pembangkit peka pada variabel cuaca yang berbeda, sehingga tiap kategori diwakili satu titik koordinat:

| Kategori | Jenis PLT | Titik koordinat | Dasar penentuan | Parameter NASA POWER |
|---|---|---|---|---|
| **Hydro** | PLTA, PLTM | −2,86 ; 120,79 | centroid tertimbang kapasitas 7 PLTA besar (Luwu Timur/Luwu) | `PRECTOTCORR` — curah hujan |
| **Solar** | PLTS, PLTS Atap | −5,10 ; 119,60 | titik representatif provinsi (Makassar/Maros) | `ALLSKY_SFC_SW_DWN` — radiasi |
| **Wind** | PLTB | −4,64 ; 119,82 | centroid tertimbang PLTB Sidrap + Jeneponto | `WS10M` — kecepatan angin 10 m |

PLTS dan PLTS Atap tersebar di lebih dari 45 lokasi kecil, sehingga dipakai satu titik representatif provinsi — penyederhanaan yang perlu dicatat sebagai keterbatasan.

In [ ]:
LOKASI = {
    "Hydro": {"lat": -2.86, "lon": 120.79, "parameter": "PRECTOTCORR"},
    "Solar": {"lat": -5.10, "lon": 119.60, "parameter": "ALLSKY_SFC_SW_DWN"},
    "Wind":  {"lat": -4.64, "lon": 119.82, "parameter": "WS10M"},
}

# Tiap jenis PLT mewarisi cuaca dari kategori induknya.
PETA_KATEGORI = {
    "PLTA": "Hydro", "PLTM": "Hydro",
    "PLTS": "Solar", "PLTS Atap": "Solar",
    "PLTB": "Wind",
}

pd.DataFrame([
    {"Kategori": k, "Lintang": v["lat"], "Bujur": v["lon"], "Parameter": v["parameter"],
     "Jenis PLT": ", ".join([j for j, kat in PETA_KATEGORI.items() if kat == k])}
    for k, v in LOKASI.items()
])

---
## Tahap 3 — Menarik Data Cuaca Riil dari NASA POWER

[NASA POWER](https://power.larc.nasa.gov/) menyediakan data meteorologi satelit gratis tanpa perlu kunci API.

Kolom Cuaca adalah **satu-satunya kolom observasi riil** dalam dataset ini — Produksi dan Kapasitas berasal dari kalkulasi ESDM.

> Kode khusus `-999.0` berarti data hilang dan harus dilewati. Kunci `"YYYY13"` adalah rata-rata tahunan, bukan bulan ke-13 — juga dilewati.

In [ ]:
import requests

BASE_URL = "https://power.larc.nasa.gov/api/temporal/monthly/point"
TAHUN_AWAL, TAHUN_AKHIR = 2023, 2025


def tarik_cuaca(nama_kategori, lat, lon, parameter, mulai=TAHUN_AWAL, sampai=TAHUN_AKHIR):
    # Tarik satu parameter cuaca bulanan untuk satu titik koordinat.
    resp = requests.get(BASE_URL, params={
        "parameters": parameter, "community": "RE",
        "longitude": lon, "latitude": lat, "format": "JSON",
        "start": mulai, "end": sampai,
    }, timeout=60)
    resp.raise_for_status()
    bulanan = resp.json()["properties"]["parameter"][parameter]

    baris = []
    for kunci, nilai in bulanan.items():
        tahun, bulan = kunci[:4], kunci[4:6]
        if bulan == "13" or nilai == -999.0:   # rata-rata tahunan / data hilang
            continue
        baris.append({
            "Tahun": int(tahun), "Bulan": int(bulan),
            "Kategori": nama_kategori, "Parameter_Cuaca": parameter,
            "Nilai_Cuaca": nilai, "Lat": lat, "Lon": lon,
        })
    return baris


semua = []
for kategori, info in LOKASI.items():
    print(f"Menarik {kategori} ({info['parameter']}) di ({info['lat']}, {info['lon']})...")
    hasil = tarik_cuaca(kategori, info["lat"], info["lon"], info["parameter"])
    semua.extend(hasil)
    print(f"   -> {len(hasil)} baris")

cuaca = pd.DataFrame(semua).sort_values(["Kategori", "Tahun", "Bulan"]).reset_index(drop=True)
cuaca.to_csv("cuaca_riil_regional.csv", index=False)

print(f"\nTotal {len(cuaca)} baris (harus 36 per kategori untuk 2023-2025):")
print(cuaca.groupby("Kategori").size().to_string())
cuaca.head()

### Pemeriksaan: apakah data cuaca masuk akal?

Sebelum dipakai, nilainya diperiksa terhadap rentang wajar iklim tropis Sulawesi Selatan.

In [ ]:
periksa = cuaca.groupby(["Kategori", "Parameter_Cuaca"])["Nilai_Cuaca"].agg(["min", "mean", "max"]).round(3)
print(periksa.to_string())
print()
print("Rentang yang diharapkan:")
print("  Hydro (curah hujan, mm/hari) : 0 - 20   -> pola musim hujan/kemarau jelas")
print("  Solar (radiasi, kWh/m2/hari) : 3 - 7    -> stabil sepanjang tahun (tropis)")
print("  Wind  (angin, m/s)           : 1 - 6    -> relatif rendah, khas Sulsel")

---
## Tahap 4 — Disagregasi: Tahunan → Bulanan

**Inilah tahap yang membentuk dataset utuh.** Dinas ESDM hanya menyediakan angka **tahunan**, sedangkan model butuh deret **bulanan**. Cuaca riil dipakai sebagai penimbang untuk membagi angka tahunan ke 12 bulan.

$$\text{Produksi}_{bulan} = \text{Produksi}_{tahun} \times \frac{\text{Basis}_{bulan}}{\sum_{12} \text{Basis}}$$

**Perbedaan perlakuan antar kategori** — ini keputusan metodologis penting:

| Kategori | Basis proporsi | Alasan fisik |
|---|---|---|
| **Hydro** (PLTA, PLTM) | rata-rata bergerak **3 bulan** dari curah hujan | waduk menyimpan air — produksi bulan ini dipengaruhi hujan bulan-bulan sebelumnya |
| **Solar & Wind** | nilai cuaca bulan itu sendiri | respons instan, tidak ada penyimpanan energi |

Kapasitas tidak didisagregasi — nilainya tetap sepanjang tahun (kapasitas terpasang).

In [ ]:
KATEGORI_PAKAI_LAG = {"Hydro"}
JENDELA_RATA_RATA_BERGERAK = 3      # bulan berjalan + 2 bulan sebelumnya
TOLERANSI_VALIDASI = 0.01           # GWh


def hitung_basis_proporsi(sub, kategori):
    # Hydro pakai rata-rata bergerak (proksi tampungan waduk); lainnya langsung.
    if kategori in KATEGORI_PAKAI_LAG:
        return sub["Nilai_Cuaca"].rolling(
            window=JENDELA_RATA_RATA_BERGERAK, min_periods=1
        ).mean()
    return sub["Nilai_Cuaca"]


def disagregasi(cuaca, annual):
    baris = []
    for tahun, jenis_dict in annual.items():
        for jenis, angka in jenis_dict.items():
            kategori = PETA_KATEGORI[jenis]
            sub = cuaca[(cuaca["Kategori"] == kategori) & (cuaca["Tahun"] == tahun)]
            if sub.empty:
                print(f"  [PERINGATAN] cuaca {jenis} ({kategori}) {tahun} tidak ada, dilewati")
                continue

            sub = sub.sort_values("Bulan").reset_index(drop=True).copy()
            sub["Basis"] = hitung_basis_proporsi(sub, kategori)
            sub["Proporsi"] = sub["Basis"] / sub["Basis"].sum()
            sub["Produksi_Bulanan"] = sub["Proporsi"] * angka["Produksi"]

            for _, r in sub.iterrows():
                baris.append({
                    "Tanggal": f"{tahun}-{int(r['Bulan']):02d}-01",
                    "Produksi": round(r["Produksi_Bulanan"], 4),
                    "Kapasitas": angka["Kapasitas"],
                    "Cuaca": r["Nilai_Cuaca"],
                    "Jenis": jenis,
                })
    return pd.DataFrame(baris).sort_values(["Jenis", "Tanggal"]).reset_index(drop=True)


df = disagregasi(cuaca, ANNUAL)
df.to_csv("DATA_REGIONAL_5JENIS.csv", index=False)

print(f"Dataset terbentuk: {len(df)} baris "
      f"({df['Jenis'].nunique()} jenis PLT x 3 tahun x 12 bulan)")
df.head(12)

### Validasi wajib: jumlah bulanan harus sama persis dengan angka tahunan

Disagregasi hanya **membagi ulang** angka tahunan — tidak boleh menambah atau mengurangi total. Kalau ada satu saja yang gagal, seluruh dataset tidak sah dipakai.

In [ ]:
def validasi(df_final, annual):
    semua_lolos, laporan = True, []
    for tahun, jenis_dict in annual.items():
        for jenis, angka in jenis_dict.items():
            subset = df_final[(df_final["Jenis"] == jenis)
                              & (df_final["Tanggal"].str.startswith(str(tahun)))]
            if subset.empty:
                continue
            total, target = subset["Produksi"].sum(), angka["Produksi"]
            selisih = abs(total - target)
            lolos = selisih < TOLERANSI_VALIDASI
            semua_lolos &= lolos
            laporan.append({"Tahun": tahun, "Jenis": jenis, "Hasil": round(total, 4),
                            "Target": round(target, 4), "Selisih": round(selisih, 6),
                            "Status": "OK" if lolos else "GAGAL"})
    return semua_lolos, pd.DataFrame(laporan)


lolos, laporan = validasi(df, ANNUAL)
print("HASIL VALIDASI:", "SEMUA LOLOS" if lolos else "ADA YANG GAGAL - CEK ULANG")
laporan

> **Alternatif:** kalau ingin langsung memakai dataset yang sudah divalidasi dari repositori (tanpa menarik ulang NASA POWER), unggah `DATA_REGIONAL_5JENIS.csv` ke Colab lalu jalankan `df = pd.read_csv("DATA_REGIONAL_5JENIS.csv")`.

---
## Tahap 5 — Framing Target & Rekayasa Fitur

### Mengapa target diganti dari GWh ke *Capacity Factor*?

Ini keputusan paling menentukan dalam penelitian ini. Tiga alasannya:

1. **Menyamakan skala.** PLTA berproduksi ratusan GWh, PLTS Atap hanya ~0,1 GWh. Satu model pooled tidak bisa belajar keduanya dalam satuan GWh.
2. **Menghapus lompatan palsu.** Kenaikan produksi 2025 sebagian besar hanyalah penambahan kapasitas terpasang, bukan perubahan pola. Membagi dengan kapasitas menghilangkan efek itu.
3. **Mengatasi negative transfer** saat pre-training memakai data nasional.

$$\text{CF} = \frac{\text{Produksi (GWh)}}{\text{Kapasitas (MW)}}$$

> Perhatikan: rumus ini **rasio sederhana tanpa faktor jam**. Jangan tertukar dengan `Produksi × 1000 / (Kapasitas × jam)` dari versi pipeline lama — skalanya berbeda.

### Varian penanganan cuaca yang diuji

| Varian | Perlakuan cuaca | Butuh info bulan target? |
|---|---|---|
| **A** | hanya lag `t-w..t-1` | Tidak — peramalan murni |
| **B** | A + cuaca bulan-`t` di lapisan Dense | Ya |
| **C** | kanal cuaca digeser → `t-w+1..t` (*known-future covariate*) | Ya |
| **D** | C + encoding siklik bulan (sin/cos) | Ya |
| **E** | C + rasio cuaca relatif terhadap rata-rata 12 bulan | Ya |
| **F** | E + encoding bulan | Ya |

> ### ⚠️ Kualifikasi yang wajib diungkap
> Varian B–F melihat cuaca bulan target. Karena kolom Produksi **dibangun dari** cuaca, sebagian keunggulannya adalah **artefak konstruksi dataset**, bukan murni kemampuan prediksi.
>
> Agar dipakai sungguhan, cuaca bulan target harus datang dari **prakiraan** atau **normal klimatologis** — bukan nilai aktual seperti dalam eksperimen ini. Sistem web SIPREBAR sudah menerapkan hal ini (NASA POWER untuk bulan lampau, klimatologis untuk bulan depan).

In [ ]:
df["Tanggal"] = pd.to_datetime(df["Tanggal"])
df["Tahun"] = df["Tanggal"].dt.year
df["Bulan"] = df["Tanggal"].dt.month
df["CF"] = df["Produksi"] / df["Kapasitas"]          # target model
df["bulan_sin"] = np.sin(2 * np.pi * df["Bulan"] / 12)
df["bulan_cos"] = np.cos(2 * np.pi * df["Bulan"] / 12)

# Rasio cuaca relatif: kausal -- hanya memakai bulan t dan sebelumnya.
df = df.sort_values(["Jenis", "Tanggal"]).reset_index(drop=True)
df["cuaca_rel"] = df.groupby("Jenis")["Cuaca"].transform(
    lambda s: s / s.rolling(12, min_periods=1).mean()
)

VARIAN = {                    # varian -> (fitur eksogen di Dense, geser kanal cuaca)
    "A": ([], False),
    "B": (["cuaca_t"], False),
    "C": ([], True),
    "D": (["bulan_sin", "bulan_cos"], True),
    "E": (["cuaca_rel"], True),
    "F": (["cuaca_rel", "bulan_sin", "bulan_cos"], True),
}

print("Rentang CF per jenis PLT (memperlihatkan kenapa skala perlu disamakan):")
print(df.groupby("Jenis")[["Produksi", "CF"]].agg(["min", "max"]).round(4).to_string())

### Pembagian data — aturan yang tidak boleh dilanggar

| Bagian | Periode | Kegunaan |
|---|---|---|
| Latih | 2023–2024 | melatih model & fit scaler |
| **Uji** | **2025** | **hanya** untuk melaporkan hasil akhir |

> **Data uji 2025 TIDAK PERNAH** menyentuh pemilihan konfigurasi maupun fit scaler. Pemilihan konfigurasi murni dari skor *walk-forward* pada 2023–2024. Melanggar aturan ini disebut **data leakage** dan membuat seluruh angka evaluasi tidak sah.

In [ ]:
TAHUN_AKHIR_TRAIN = 2024
TAHUN_TEST = 2025

def buat_sequence(data, window, geser_cuaca, kolom_exog):
    # Bentuk sequence per jenis PLT. Kanal: [CF, Cuaca].
    # geser_cuaca=True -> kanal cuaca memuat t-w+1..t (known-future covariate),
    # sementara kanal CF tetap t-w..t-1.
    X, y, kat, exog, tanggal = [], [], [], [], []
    for idx_k, jenis in enumerate(JENIS_PLT):
        s = data[data["Jenis"] == jenis].sort_values("Tanggal").reset_index(drop=True)
        for i in range(window, len(s)):
            cf_lag = s["CF"].values[i - window:i]
            if geser_cuaca:
                cuaca_win = s["Cuaca"].values[i - window + 1:i + 1]
            else:
                cuaca_win = s["Cuaca"].values[i - window:i]
            X.append(np.column_stack([cf_lag, cuaca_win]))
            y.append(s["CF"].values[i])
            kat.append(idx_k)
            baris_exog = {"cuaca_t": s["Cuaca"].values[i],
                          "cuaca_rel": s["cuaca_rel"].values[i],
                          "bulan_sin": s["bulan_sin"].values[i],
                          "bulan_cos": s["bulan_cos"].values[i]}
            exog.append([baris_exog[k] for k in kolom_exog] if kolom_exog else [])
            tanggal.append(s["Tanggal"].values[i])
    return {"X": np.array(X), "y": np.array(y), "kat": np.array(kat),
            "exog": np.array(exog) if kolom_exog else None,
            "tanggal": pd.to_datetime(tanggal)}

contoh = buat_sequence(df, window=3, geser_cuaca=True, kolom_exog=[])
print("Bentuk sequence (window=3):", contoh["X"].shape, "-> (n_sampel, window, [CF, Cuaca])")
print("Total sampel :", len(contoh['y']))
print("Sampel latih :", (contoh['tanggal'].year <= TAHUN_AKHIR_TRAIN).sum())
print("Sampel uji   :", (contoh['tanggal'].year == TAHUN_TEST).sum())

---
## Tahap 6 — Arsitektur Model

### Satu model *pooled* untuk semua jenis PLT

Alih-alih 5 model terpisah, dipakai **satu model bersama**; identitas jenis PLT masuk lewat **category embedding**. Alasannya: dengan hanya 24 titik latih per jenis PLT, melatih model terpisah membuat sampel efektif terlalu sedikit. Pooling menaikkan sampel latih berlipat ganda.

```
Input sequence (window, 2)          Input kategori (1,)
        │                                   │
    LSTM(64, return_sequences)        Embedding(5 → 4)
        │                                   │
    Dropout(0.2)                        Flatten
        │                                   │
    LSTM(32)                                │
        │                                   │
    Dropout(0.2)                            │
        └──────────┬────────────────────────┘
                   │  (+ fitur eksogen bila varian memakainya)
              Concatenate
                   │
               Dense(1) → CF
```

Arsitektur LSTM **64 → 32 → Dense(1)** sengaja dipertahankan identik dengan pipeline pembanding, supaya yang diuji adalah **framing**-nya, bukan kapasitas modelnya.

In [ ]:
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, Embedding, Flatten, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

N_SEED = 3              # ensembling: 3 seed, prediksi dirata-rata
EPOCHS = 200
BATCH_SIZE = 8
DROPOUT = 0.2

KANDIDAT_KONFIG = [
    {"window": 3, "lr": 1e-3},
    {"window": 3, "lr": 1e-4},
    {"window": 6, "lr": 1e-3},
    {"window": 6, "lr": 1e-4},
]


def bangun_model(window, n_fitur, dropout, lr, n_kategori, n_exog):
    # Model pooled; identitas jenis PLT masuk lewat embedding.
    in_seq = Input(shape=(window, n_fitur), name="sekuens")
    in_kat = Input(shape=(1,), dtype="int32", name="kategori")

    x = LSTM(64, return_sequences=True)(in_seq)
    x = Dropout(dropout)(x)
    x = LSTM(32)(x)
    x = Dropout(dropout)(x)

    emb = Flatten()(Embedding(n_kategori, 4, name="embed_kategori")(in_kat))

    inputs, gabung = [in_seq, in_kat], [x, emb]
    if n_exog:
        in_exog = Input(shape=(n_exog,), name="eksogen")
        inputs.append(in_exog)
        gabung.append(in_exog)

    out = Dense(1)(Concatenate()(gabung))
    model = Model(inputs=inputs, outputs=out)
    model.compile(optimizer=Adam(learning_rate=lr), loss="mse", metrics=["mae"])
    return model


demo = bangun_model(window=3, n_fitur=2, dropout=DROPOUT, lr=1e-3, n_kategori=5, n_exog=0)
demo.summary()

### Metrik evaluasi

**RMSE** dipakai sebagai metrik utama (satuan GWh, sebanding langsung dengan produksi), didampingi MAE dan MAPE.

Seluruh metrik dilaporkan kembali dalam **GWh** (CF × Kapasitas), bukan dalam skala CF — supaya angkanya bermakna bagi pembaca dan sebanding dengan benchmark.

In [ ]:
def rmse(y, yhat):  return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(yhat)) ** 2)))
def mae(y, yhat):   return float(np.mean(np.abs(np.asarray(y) - np.asarray(yhat))))

def mape(y, yhat):
    y, yhat = np.asarray(y, float), np.asarray(yhat, float)
    m = y != 0                      # hindari pembagian nol
    return float(np.mean(np.abs((y[m] - yhat[m]) / y[m])) * 100)

def metrik(y, yhat):
    return {"RMSE": rmse(y, yhat), "MAE": mae(y, yhat), "MAPE": mape(y, yhat)}

---
## Tahap 7 — Walk-Forward Validation & Pemilihan Konfigurasi

### Mengapa bukan split 85/15 biasa?

Dengan hanya belasan sequence, satu split acak sangat berisik — hasilnya bisa berubah drastis hanya karena kebetulan pembagian. **Walk-forward (rolling-origin)** melatih berulang dengan titik potong bergerak maju, sehingga skornya jauh lebih stabil dan meniru cara model dipakai sungguhan (selalu meramal ke depan).

### Aturan emas pemilihan konfigurasi

> Konfigurasi pemenang dipilih **murni dari skor walk-forward pada 2023–2024**. Data uji 2025 tidak dilibatkan sama sekali. Kombinasi **tidak boleh ditukar** hanya karena skor 2025-nya kebetulan lebih bagus — itu *test-set leakage*.

Pemilihan dilakukan **per jenis PLT**, bukan satu pemenang global. Alasannya: skor global adalah rata-rata RMSE dalam GWh, sehingga didominasi PLTA (ratusan GWh) dan praktis mengabaikan PLTS (0,1 GWh).

In [ ]:
def lipatan_walk_forward(tanggal_train, n_lipat=4, ukuran_val=3):
    # Hasilkan (indeks_latih, indeks_validasi) dengan titik potong bergerak maju.
    urut = np.argsort(tanggal_train)
    n = len(urut)
    lipatan = []
    for k in range(n_lipat):
        akhir_val = n - k * ukuran_val
        awal_val = akhir_val - ukuran_val
        if awal_val <= ukuran_val:
            break
        lipatan.append((urut[:awal_val], urut[awal_val:akhir_val]))
    return list(reversed(lipatan))


contoh_tgl = contoh["tanggal"][contoh["tanggal"].year <= TAHUN_AKHIR_TRAIN]
lipatan = lipatan_walk_forward(np.array(contoh_tgl))
print(f"Jumlah lipatan walk-forward: {len(lipatan)}\n")
for i, (tr, va) in enumerate(lipatan, 1):
    print(f"  Lipatan {i}: latih={len(tr):3d} sampel  |  validasi={len(va):2d} sampel")

### Menjalankan pencarian konfigurasi

Sel berikut menjalankan seluruh kombinasi **varian × konfigurasi** dengan walk-forward.

> ⏱️ **Perhatian waktu:** 6 varian × 4 konfigurasi × 4 lipatan × 3 seed ≈ **288 pelatihan**. Di CPU Colab ini bisa memakan **30–60 menit**.
>
> Untuk demo cepat kepada pembimbing, set `MODE_CEPAT = True` — hanya menguji varian B dan C dengan 1 seed. Hasil akhirnya tidak identik dengan skripsi, tetapi alurnya persis sama. Untuk mereproduksi angka resmi, pakai `MODE_CEPAT = False`.

In [ ]:
MODE_CEPAT = True     # ← ubah ke False untuk reproduksi penuh angka skripsi

if MODE_CEPAT:
    varian_diuji, konfig_diuji, n_seed_pakai, epochs_pakai = ["B", "C"], KANDIDAT_KONFIG[:2], 1, 60
    print("MODE CEPAT — untuk demo alur. Angka tidak identik dengan skripsi.\n")
else:
    varian_diuji, konfig_diuji, n_seed_pakai, epochs_pakai = list(VARIAN), KANDIDAT_KONFIG, N_SEED, EPOCHS
    print("MODE PENUH — mereproduksi angka skripsi. Butuh 30-60 menit.\n")


def skala_cf(train_vals):
    # Fit scaler HANYA dari data latih. Mengembalikan (min, max).
    return float(np.min(train_vals)), float(np.max(train_vals))

def terapkan(v, lo, hi):   return (v - lo) / (hi - lo) if hi > lo else v * 0.0
def balikkan(v, lo, hi):   return v * (hi - lo) + lo


def siapkan_input(seq, idx, kolom_exog):
    x = [seq["X"][idx], seq["kat"][idx]]
    if kolom_exog:
        x.append(seq["exog"][idx])
    return x


def latih_satu(seq, idx_tr, idx_va, window, konfig, kolom_exog, seed, epochs):
    tf.keras.backend.clear_session()
    tf.random.set_seed(seed); np.random.seed(seed)
    m = bangun_model(window, seq["X"].shape[2], DROPOUT, konfig["lr"], len(JENIS_PLT), len(kolom_exog))
    cb = [EarlyStopping(monitor="loss", patience=20, restore_best_weights=True)]
    m.fit(siapkan_input(seq, idx_tr, kolom_exog), seq["y_skala"][idx_tr],
          epochs=epochs, batch_size=BATCH_SIZE, callbacks=cb, verbose=0)
    return m


hasil_wf = []
for nama_varian in varian_diuji:
    kolom_exog, geser = VARIAN[nama_varian]
    for konfig in konfig_diuji:
        seq = buat_sequence(df, konfig["window"], geser, kolom_exog)
        mask_tr = seq["tanggal"].year <= TAHUN_AKHIR_TRAIN
        idx_train_all = np.where(mask_tr)[0]

        lo, hi = skala_cf(seq["y"][idx_train_all])       # scaler dari data latih saja
        seq["y_skala"] = terapkan(seq["y"], lo, hi)

        skor_per_plt = {j: [] for j in JENIS_PLT}
        for idx_tr_rel, idx_va_rel in lipatan_walk_forward(np.array(seq["tanggal"][idx_train_all])):
            idx_tr, idx_va = idx_train_all[idx_tr_rel], idx_train_all[idx_va_rel]
            pred_seed = []
            for s in range(n_seed_pakai):
                m = latih_satu(seq, idx_tr, idx_va, konfig["window"], konfig, kolom_exog, SEED + s, epochs_pakai)
                pred_seed.append(m.predict(siapkan_input(seq, idx_va, kolom_exog), verbose=0).ravel())
            pred_cf = balikkan(np.mean(pred_seed, axis=0), lo, hi)
            asli_cf = seq["y"][idx_va]

            for idx_k, jenis in enumerate(JENIS_PLT):
                sel = seq["kat"][idx_va] == idx_k
                if sel.sum() == 0:
                    continue
                kap = df[df["Jenis"] == jenis]["Kapasitas"].median()
                skor_per_plt[jenis].append(rmse(asli_cf[sel] * kap, pred_cf[sel] * kap))

        for jenis, skor in skor_per_plt.items():
            if skor:
                hasil_wf.append({"Varian": nama_varian, "window": konfig["window"],
                                 "lr": konfig["lr"], "Jenis": jenis,
                                 "RMSE_walkforward": float(np.mean(skor))})
        print(f"  selesai: varian {nama_varian}, window={konfig['window']}, lr={konfig['lr']}")

wf = pd.DataFrame(hasil_wf)
print(f"\n{len(wf)} baris skor walk-forward terkumpul.")
wf.head(10)

In [ ]:
# Pemenang per jenis PLT = skor walk-forward terkecil (BUKAN skor data uji).
pemenang = wf.loc[wf.groupby("Jenis")["RMSE_walkforward"].idxmin()].reset_index(drop=True)
print("Konfigurasi terpilih per jenis PLT (dari walk-forward 2023-2024):")
pemenang[["Jenis", "Varian", "window", "lr", "RMSE_walkforward"]].round(4)

### Konfigurasi pemenang resmi (hasil MODE PENUH)

Kalau Anda menjalankan `MODE_CEPAT = True`, tabel di atas akan berbeda. Berikut konfigurasi resmi yang dipakai sistem SIPREBAR:

| Jenis PLT | Varian | Window | Learning rate | Berkas model |
|---|---|---|---|---|
| PLTA | B | 3 | 1e−3 | `B_w3_lr1e-3_seed{0,1,2}.keras` |
| PLTB | B | **6** | 1e−3 | `B_w6_lr1e-3_seed{0,1,2}.keras` |
| PLTM | B | 3 | 1e−4 | `B_w3_lr1e-4_seed{0,1,2}.keras` |
| PLTS & PLTS Atap | C | 3 | 1e−3 | `C_w3_lr1e-3_seed{0,1,2}.keras` *(berbagi)* |

**PLTS dan PLTS Atap memakai berkas model yang sama** — dibedakan lewat *category embedding*. Keduanya adalah deret yang identik proporsional (cuaca sama, rasio produksi/kapasitas sama), sehingga wajar berbagi konfigurasi.

Perbedaan window inilah yang menjawab pertanyaan *"kenapa ada yang 6 bulan dan ada yang 3 bulan"* — tiap kategori punya konfigurasi pemenangnya sendiri.

---
## Tahap 8 — Benchmark: Metode Tradisional

Model LSTM harus dibandingkan dengan metode sederhana yang **pantas**. Untuk data bulanan musiman, naive lag-1 saja tidak cukup — perlu benchmark musiman.

| Benchmark | Rumus | Logika |
|---|---|---|
| **Naive lag-1** | $\hat{y}_t = y_{t-1}$ | "bulan ini sama dengan bulan lalu" |
| **Seasonal naive lag-12** | $\hat{y}_t = y_{t-12}$ | "sama dengan bulan yang sama tahun lalu" |
| **Seasonal × rasio kapasitas** | $\hat{y}_t = y_{t-12} \times \frac{K_t}{K_{t-12}}$ | seasonal naive, disesuaikan pertumbuhan kapasitas |
| **ARIMA(1,1,1)** | model statistik klasik | pembanding akademik standar |

Benchmark ketiga penting karena kapasitas terpasang Sulsel bertumbuh tiap tahun — seasonal naive polos akan selalu *under-predict*.

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

def hitung_benchmark(data, jenis):
    s = data[data["Jenis"] == jenis].sort_values("Tanggal").reset_index(drop=True)
    y_test = s.loc[s["Tahun"] == TAHUN_TEST, "Produksi"].values
    y_2024 = s.loc[s["Tahun"] == 2024, "Produksi"].values
    kap_test = s.loc[s["Tahun"] == TAHUN_TEST, "Kapasitas"].values
    kap_2024 = s.loc[s["Tahun"] == 2024, "Kapasitas"].values
    y_latih = s.loc[s["Tahun"] <= TAHUN_AKHIR_TRAIN, "Produksi"].values

    hasil = {}
    # Naive lag-1: bulan Des 2024 jadi prediksi Jan 2025, dst.
    seri = s["Produksi"].values
    idx_test = np.where(s["Tahun"].values == TAHUN_TEST)[0]
    hasil["Naive_lag1"] = metrik(y_test, seri[idx_test - 1])
    hasil["SeasonalNaive_lag12"] = metrik(y_test, y_2024)
    hasil["SeasonalNaive_x_rasio_kapasitas"] = metrik(y_test, y_2024 * (kap_test / kap_2024))

    try:
        pred_arima = ARIMA(y_latih, order=(1, 1, 1)).fit().forecast(steps=len(y_test))
        hasil["ARIMA(1,1,1)"] = metrik(y_test, pred_arima)
    except Exception as e:
        print(f"  ARIMA gagal untuk {jenis}: {e}")
    return hasil


baris_bm = []
for jenis in JENIS_PLT:
    for nama, m in hitung_benchmark(df, jenis).items():
        baris_bm.append({"Jenis": jenis, "Metode": nama, **m})

benchmark_df = pd.DataFrame(baris_bm)
benchmark_df.round(4)

---
## Tahap 9 — Hasil Evaluasi Akhir

Angka berikut adalah hasil resmi penelitian (MODE PENUH, data uji 2025, satuan GWh). Inilah yang ditampilkan halaman **Gap Analysis RUED** pada sistem web.

In [ ]:
HASIL_RESMI = pd.DataFrame([
    # Jenis,        LSTM,    ARIMA,   Naive lag-1, Seasonal x Kapasitas
    ["PLTA",       77.062,  117.660,  90.976,  98.278],
    ["PLTB",        5.071,   11.866,  10.423,   9.050],
    ["PLTM",        5.831,   10.309,   7.497,   8.592],
    ["PLTS",        0.0958,   0.5480,  0.1363,  0.0801],
    ["PLTS Atap",   0.1138,   0.5668,  0.1509,  0.0960],
], columns=["Jenis PLT", "LSTM (produksi)", "ARIMA(1,1,1)", "Naive lag-1", "Seasonal × Kapasitas"])

metode_kol = ["LSTM (produksi)", "ARIMA(1,1,1)", "Naive lag-1", "Seasonal × Kapasitas"]
HASIL_RESMI["Terbaik"] = HASIL_RESMI[metode_kol].idxmin(axis=1)

menang = (HASIL_RESMI["Terbaik"] == "LSTM (produksi)").sum()
print(f"LSTM unggul di {menang} dari {len(HASIL_RESMI)} jenis PLT\n")
HASIL_RESMI

In [ ]:
MAPE_RESMI = {"PLTA": 10.99, "PLTB": 7.49, "PLTM": 8.90, "PLTS": 7.90, "PLTS Atap": 7.92}
akurasi = pd.DataFrame([
    {"Jenis PLT": j, "MAPE (%)": m, "Akurasi (100-MAPE) (%)": round(100 - m, 2)}
    for j, m in MAPE_RESMI.items()
])
print("Akurasi model produksi pada data uji 2025:")
akurasi

### 📌 Kesimpulan penelitian

**LSTM unggul di 3 dari 5 jenis PLT** (PLTA, PLTB, PLTM), dengan **akurasi 89–93% di kelima kategori**.

PLTS dan PLTS Atap **kalah tipis** dari *seasonal naive × rasio kapasitas*.

### ⚠️ Catatan integritas — bagian terpenting untuk dipertahankan di sidang

> Kombinasi model untuk PLTS/PLTS Atap **tidak ditukar** ke varian lain yang skor data ujinya kebetulan lebih bagus.
>
> Sebelum ensembling, LSTM sempat unggul di **4 dari 5** — varian E memenangkan PLTS Atap. Varian E memang tetap mengalahkan seasonal naive pada data uji 2025 (RMSE 0,075 vs 0,096), **tetapi** skor walk-forward-nya (0,0305) kalah dari varian C (0,0164).
>
> Menukar ke varian E berarti memilih berdasarkan skor data uji — itu **data leakage**, pelanggaran yang sama persis dengan temuan yang sudah diperbaiki di tahap audit sebelumnya. Maka varian C dipertahankan, dan kekalahannya dilaporkan apa adanya.
>
> **Kesimpulan jujur:** pada 24 titik latih per jenis PLT, pemilihan varian untuk PLTS/PLTS Atap masih berada di dalam rentang ketidakpastian. Ini keterbatasan ukuran data, **bukan bukti bahwa LSTM tidak mampu**.

Sistem web SIPREBAR memakai **LSTM untuk kelima jenis PLT secara konsisten**, termasuk PLTS dan PLTS Atap. Metode tradisional hanya berperan sebagai pembanding di tabel evaluasi, tidak pernah dipakai untuk melayani prediksi.

---
## Tahap 10 — Menyimpan Artefak Model

Model final disimpan sebagai `.keras`, dan parameter penskalaan sebagai JSON. Backend memuat berkas inilah saat melayani prediksi — backend **tidak pernah melatih ulang**.

In [ ]:
from google.colab import files    # hapus baris ini bila dijalankan di luar Colab

df.to_csv("DATA_REGIONAL_5JENIS.csv", index=False)
benchmark_df.to_csv("benchmark_2025.csv", index=False)
HASIL_RESMI.to_csv("evaluasi_final.csv", index=False)

konfigurasi_produksi = {
    "PLTA":      {"varian": "B", "window": 3, "lr": 1e-3},
    "PLTB":      {"varian": "B", "window": 6, "lr": 1e-3},
    "PLTM":      {"varian": "B", "window": 3, "lr": 1e-4},
    "PLTS":      {"varian": "C", "window": 3, "lr": 1e-3},
    "PLTS Atap": {"varian": "C", "window": 3, "lr": 1e-3},
}
with open("konfigurasi_model.json", "w") as f:
    json.dump(konfigurasi_produksi, f, indent=2)

print("Berkas siap diunduh:")
for f_ in ["DATA_REGIONAL_5JENIS.csv", "benchmark_2025.csv",
           "evaluasi_final.csv", "konfigurasi_model.json"]:
    print("  -", f_)

# files.download("DATA_REGIONAL_5JENIS.csv")   # aktifkan untuk mengunduh

---
## Ringkasan Alur

```
Data tahunan ESDM (5 jenis PLT, 2023-2025)
         │
         ├── NASA POWER ──> cuaca_riil_regional.csv   (observasi riil)
         │
         ▼
    DISAGREGASI (cuaca sebagai penimbang)
    Hydro: rata-rata bergerak 3 bulan · Solar/Wind: langsung
         │
         ▼
    DATA_REGIONAL_5JENIS.csv  (180 baris — dataset utuh)
         │
         ▼
    Target -> Capacity Factor  +  6 varian fitur
         │
         ▼
    WALK-FORWARD (2023-2024)  ──> pilih konfigurasi per jenis PLT
         │                          (data uji 2025 TIDAK dilibatkan)
         ▼
    Model pooled + embedding + ensembling 3 seed
         │
         ▼
    EVALUASI pada 2025  ──> LSTM unggul 3 dari 5, akurasi 89-93%
         │
         ▼
    Artefak .keras + konfigurasi_model.json
         │
         ▼
    [ Backend FastAPI ] ──> [ Frontend React ]
```

---

### Berkas terkait di repositori

| Berkas | Isi |
|---|---|
| `audit/source/tarik_cuaca_nasa_power.py` | Tahap 3 — penarikan cuaca |
| `audit/source/disagregasi_regional.py` | Tahap 4 — disagregasi |
| `audit/analysis/lstm_improved.py` | Tahap 5–7 — eksperimen framing & walk-forward |
| `audit/analysis/build_production_model.py` | Tahap 7 — model produksi final |
| `audit/analysis/baseline_compare.py` | Tahap 8 — benchmark |
| `audit/results/framing_findings.md` | Temuan lengkap & catatan kejujuran metodologis |

---

*SIPREBAR — Sistem Prediksi Energi Baru Terbarukan*
*UIN Alauddin Makassar, 2026*